In [ ]:
%load_ext autoreload
%autoreload 2

from itertools import product
import sys
sys.path.append('/home/projects/nyosef/zvise/PixelGen/')
from PixelGen.multimodalvi import MultiModalSCVI
from PixelGen.multimodalvae import MultiModalVAE, AggMethod, D
from PixelGen.enums import AggMethod, D
from PixelGen.metrics import MultiModalVIMetrics
from sklearn.preprocessing import PowerTransformer
from pathlib import Path

import anndata as ad
import torch
import scvi
import scipy
# from scvi import autotune

import seaborn as sns
import scanpy as sc
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from tqdm import tqdm

from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection

# import ray
# from ray import tune


from PixelGen.pxl_utils import train_model, get_model_latents
from PixelGen.scvi_utils import plot_losses, pca_neighbors_umap, calc_PCA
from pixelator.common.statistics import clr_transformation, dsb_normalize


from pixelator.pna.plot import molecule_rank_plot
# from pixelator.plot import molecule_rank_plot, cell_count_plot, scatter_umi_per_upia_vs_tau
# from pixelator.statistics import c
# lr_transformation
# from pixelator.analysis.normalization import dsb_normalize


import tempfile

from torch.distributions import NegativeBinomial, Normal, Poisson, MixtureSameFamily, Beta
from torch.distributions import kl_divergence as kl

print(torch.cuda.is_available())
from sklearn.decomposition import PCA

from anndata import AnnData
scvi.settings.seed = 0
print("Last run with scvi-tools version:", scvi.__version__)
sc.set_figure_params(figsize=(6, 6), frameon=False)

sns.set_theme()
torch.set_float32_matmul_precision("high")
save_dir = tempfile.TemporaryDirectory()

from utils import plot_latent, plot_gene_heatmap, plot_model_latents
from doublet_seperation import B_CD4_logfc_dict, B_CD8_logfc_dict, run_cellwise_coloc_analysis_to_disk, concat_abundance_to_adata, add_doublets_metadata
%config InlineBackend.print_figure_kwargs={"facecolor": "w"}
%config InlineBackend.figure_format="retina"
from pixelator import read_pna as read
import pickle
import scipy.sparse as sp
from scipy.sparse.csgraph import dijkstra
import anndata
import hotspot

import numpy as np
import pandas as pd
import networkx as nx
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import pandas as pd
import torch
import os
from torch_geometric.data import Data
from tqdm import tqdm  # Progress bar
import glob
import torch
from torch_geometric.nn import VGAE, GCNConv

In [ ]:
DATA_DIR = Path("/home/projects/nyosef/zvise/PxlgnProject/Data")
ANNOTATED_ADATA_PATH='/home/projects/nyosef/zvise/PixelGen/PixelGen/Data/adatas/final_adatas/adata_annotated.h5ad'

files = [f for f in DATA_DIR.rglob('*.pxl') if f.is_file()]
data = read(files)
adata=sc.read_h5ad(ANNOTATED_ADATA_PATH)

In [ ]:
output_dir = "/home/projects/nyosef/zvise/PixelGen/PixelGen/cache/GVAE/graphs"
files = glob.glob(f"{output_dir}/*.pt")
ALL_MARKERS = adata.var_names.tolist()
MARKER_TO_IDX = {m: i for i, m in enumerate(ALL_MARKERS)}

In [ ]:
load_attention_weights = True


if load_attention_weights:
    import torch
    import torch.nn as nn
    from torch_geometric.nn import VGAE, GATv2Conv
    import os

    # --- 1. CONFIGURATION (Must match training exactly) ---
    INPUT_DIM = 159       
    HIDDEN_DIM = 64       
    LATENT_DIM = 16       
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Path to your saved weights
    save_path = "/home/projects/nyosef/zvise/PixelGen/PixelGen/cache/GVAE/models/gvae_gat_model.pth"

    # --- 2. DEFINE THE ARCHITECTURE (The Fix) ---
    # You must define this class exactly as it was during training
    class VariationalGATEncoder(torch.nn.Module):
        def __init__(self, in_channels, hidden_channels, out_channels):
            super().__init__()
            # Heads=4, concat=True -> Output size is hidden_channels * 4
            # This explains why the bias in your file is 256 (64 * 4)
            self.conv1 = GATv2Conv(in_channels, hidden_channels, heads=4, concat=True)
            
            # Calculate input size for next layer
            hidden_out = hidden_channels * 4
            
            # Heads=1, concat=False -> Output size is out_channels
            self.conv_mu = GATv2Conv(hidden_out, out_channels, heads=1, concat=False)
            self.conv_logstd = GATv2Conv(hidden_out, out_channels, heads=1, concat=False)

        def forward(self, x, edge_index):
            x = self.conv1(x, edge_index).relu()
            return self.conv_mu(x, edge_index), self.conv_logstd(x, edge_index)

    # --- 3. LOAD THE MODEL ---
    print("🔄 Loading GATv2 model...")

    # A. Initialize the correct class
    encoder = VariationalGATEncoder(INPUT_DIM, HIDDEN_DIM, LATENT_DIM)
    model = VGAE(encoder)

    # B. Load the weights
    state_dict = torch.load(save_path, map_location=DEVICE, weights_only=True)
    model.load_state_dict(state_dict)

    # C. Set to Evaluation Mode
    model.to(DEVICE)
    model.eval()

    print("✅ Model loaded successfully.")
    
else:
    # --- 1. CONFIGURATION (Must match training exactly) ---
    INPUT_DIM = 159       # Protein types
    HIDDEN_DIM = 64
    LATENT_DIM = 16
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # --- 2. DEFINE ARCHITECTURE ---
    class VariationalEncoder(torch.nn.Module):
        def __init__(self, in_channels, hidden_channels, out_channels):
            super().__init__()
            self.conv1 = GCNConv(in_channels, hidden_channels)
            self.conv_mu = GCNConv(hidden_channels, out_channels)
            self.conv_logstd = GCNConv(hidden_channels, out_channels)

        def forward(self, x, edge_index):
            x = self.conv1(x, edge_index).relu()
            return self.conv_mu(x, edge_index), self.conv_logstd(x, edge_index)

    # --- 3. INITIALIZE AND LOAD ---
    # Initialize the empty model
    encoder = VariationalEncoder(INPUT_DIM, HIDDEN_DIM, LATENT_DIM)
    model = VGAE(encoder).to(DEVICE)

    # Load the weights
    try:
        # Adjust filename if you named it something else
        model.load_state_dict(torch.load("/home/projects/nyosef/zvise/PixelGen/PixelGen/cache/GVAE/models/vgae_model_epoch6.pth"))
        model.eval() # CRITICAL: Switch to evaluation mode
        print("✅ Model loaded successfully and set to eval mode.")
    except FileNotFoundError:
        print("❌ Error: Could not find 'vgae_model_epoch6.pth'. Check your file path.")

# ATTENTION VAE

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import random_split
from torch_geometric.data import Dataset, Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GlobalAttention
from torch_geometric.utils import scatter
import numpy as np
import pandas as pd
import scanpy as sc
import os
import matplotlib.pyplot as plt
from tqdm import tqdm

# --- 1. CONFIGURATION ---
BATCH_SIZE = 32
INPUT_DIM = 16        # GVAE embedding size
LATENT_DIM = 64       # Final Cell Vector size
NUM_PROTEINS = len(MARKER_TO_IDX)
EPOCHS = 20           # Increased slightly for stable convergence
LEARNING_RATE = 1e-4  # Lower LR for stability
GRADIENT_CLIP = 5.0   # Prevents loss spikes
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Directories
CACHE_DIR = "/home/projects/nyosef/zvise/PixelGen/PixelGen/cache/GVAE/attention_model"
MODEL_DIR = "/home/projects/nyosef/zvise/PixelGen/PixelGen/cache/GVAE/attention_model/models"
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print(f"🚀 Starting Stabilized Semantic VAE Pipeline")
print(f"   • saving to: {CACHE_DIR}")

# --- 2. DATASET (Computed On-the-Fly) ---
class SemanticGraphDataset(Dataset):
    def __init__(self, file_list, gvae_model, device):
        super().__init__()
        self.file_list = file_list
        self.gvae_model = gvae_model
        self.device = device
        self.gvae_model.eval()
        self.gvae_model.to(device)

    def len(self):
        return len(self.file_list)

    def get(self, idx):
        path = self.file_list[idx]
        data = torch.load(path, weights_only=False).to(self.device)
        protein_ids = data.x.argmax(dim=1)
        with torch.no_grad():
            z_nodes = self.gvae_model.encode(data.x, data.edge_index)
        processed_data = Data(x=z_nodes.cpu(), protein_id=protein_ids.cpu(), num_nodes=data.num_nodes)
        return processed_data

# --- 3. MODEL ARCHITECTURE ---
class PerProteinVAE(nn.Module):
    def __init__(self, input_dim, latent_dim, num_proteins):
        super(PerProteinVAE, self).__init__()
        self.num_proteins = num_proteins
        self.input_dim = input_dim
        
        # Encoder
        self.att_gate = nn.Sequential(nn.Linear(input_dim, 32), nn.Tanh(), nn.Linear(32, 1))
        self.pool = GlobalAttention(gate_nn=self.att_gate)
        self.fc_mu = nn.Linear(input_dim, latent_dim)
        self.fc_logvar = nn.Linear(input_dim, latent_dim)
        
        # Decoder
        output_size = num_proteins * input_dim
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(), nn.BatchNorm1d(128),
            nn.Linear(128, 512), nn.ReLU(),
            nn.Linear(512, output_size) 
        )

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x, batch):
        h_pooled = self.pool(x, batch)
        mu = self.fc_mu(h_pooled)
        logvar = self.fc_logvar(h_pooled)
        z = self.reparameterize(mu, logvar)
        recon_flat = self.decoder(z)
        recon_matrix = recon_flat.view(-1, self.num_proteins, self.input_dim)
        return recon_matrix, mu, logvar

# --- 4. ROBUST LOSS FUNCTION ---
def semantic_loss(recon_matrix, x_embeddings, protein_ids, batch, mu, logvar):
    batch_size = batch.max().item() + 1
    total_groups = batch_size * NUM_PROTEINS
    composite_index = batch * NUM_PROTEINS + protein_ids
    
    # Scatter with explicit dim_size (Crash Fix)
    stats_sum = scatter(x_embeddings, composite_index, dim=0, dim_size=total_groups, reduce='sum')
    stats_count = scatter(torch.ones_like(composite_index), composite_index, dim=0, dim_size=total_groups, reduce='sum')
    
    stats_count = stats_count.unsqueeze(1).clamp(min=1)
    true_means = stats_sum / stats_count
    
    true_matrix = true_means.view(batch_size, NUM_PROTEINS, -1)
    mask = (stats_count.view(batch_size, NUM_PROTEINS, 1) > 0).float()
    
    mse = (recon_matrix - true_matrix) ** 2
    mse = (mse * mask).sum() / mask.sum().clamp(min=1)
    kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    
    return mse + 0.005 * kld

# --- 5. SETUP & SPLIT ---
full_dataset = SemanticGraphDataset(files, model, DEVICE)

# 80/20 Train/Validation Split
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_set, val_set = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"📊 Dataset Split: {train_size} Train | {val_size} Validation")

# --- 6. TRAINING LOOP (With Monitoring) ---
semantic_vae = PerProteinVAE(input_dim=INPUT_DIM, latent_dim=LATENT_DIM, num_proteins=NUM_PROTEINS).to(DEVICE)
optimizer = optim.Adam(semantic_vae.parameters(), lr=LEARNING_RATE)

train_history = []
val_history = []
best_val_loss = float('inf')

print("\n🔄 Training Started...")

for epoch in range(EPOCHS):
    # A. Train
    semantic_vae.train()
    train_loss = 0
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    
    for batch in loop:
        batch = batch.to(DEVICE)
        optimizer.zero_grad()
        
        recon_matrix, mu, logvar = semantic_vae(batch.x, batch.batch)
        loss = semantic_loss(recon_matrix, batch.x, batch.protein_id, batch.batch, mu, logvar)
        
        loss.backward()
        # Clip Gradients (Stability Fix)
        torch.nn.utils.clip_grad_norm_(semantic_vae.parameters(), GRADIENT_CLIP)
        
        optimizer.step()
        train_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / len(train_loader)
    train_history.append(avg_train_loss)
    
    # B. Validate
    semantic_vae.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(DEVICE)
            recon_matrix, mu, logvar = semantic_vae(batch.x, batch.batch)
            loss = semantic_loss(recon_matrix, batch.x, batch.protein_id, batch.batch, mu, logvar)
            val_loss += loss.item()
            
    avg_val_loss = val_loss / len(val_loader)
    val_history.append(avg_val_loss)
    
    print(f"   Epoch {epoch+1}: Train Loss={avg_train_loss:.4f} | Val Loss={avg_val_loss:.4f}")
    
    # Save Best Model
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(semantic_vae.state_dict(), os.path.join(MODEL_DIR, "best_semantic_vae.pth"))

print(f"✅ Training Complete. Best Val Loss: {best_val_loss:.4f}")

# --- 7. QUALITY CHECKS (Plots) ---




# A. Loss Curve
plt.figure(figsize=(10, 5))
plt.plot(train_history, label='Train Loss')
plt.plot(val_history, label='Val Loss')
plt.title('Training Stability Check')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.savefig(os.path.join(CACHE_DIR, "training_curve.png"))
plt.show()

# B. Parity Plot (Biological Reality Check)
print("\n🔬 Generating Parity Plot (Reality Check)...")
sample_batch = next(iter(val_loader)).to(DEVICE)
semantic_vae.eval()
with torch.no_grad():
    recon, _, _ = semantic_vae(sample_batch.x, sample_batch.batch)

# Get Predicted Means (Magnitude of vectors)
pred_mags = recon.norm(dim=2).cpu().numpy().flatten()

# Get True Means
batch_ids = sample_batch.batch
prot_ids = sample_batch.protein_id
composite = batch_ids * NUM_PROTEINS + prot_ids
true_sums = scatter(sample_batch.x, composite, dim=0, reduce='sum')
true_counts = scatter(torch.ones_like(composite), composite, dim=0, reduce='sum').clamp(min=1)
true_means = true_sums / true_counts.unsqueeze(1)
true_mags = true_means.norm(dim=1).cpu().numpy()

# Filter for existing proteins (count > 0)
mask = (true_counts.squeeze() > 0).cpu().numpy()
true_mags = true_mags[mask]
pred_mags = pred_mags[:len(true_mags)] # align length

plt.figure(figsize=(6, 6))
plt.scatter(true_mags, pred_mags, alpha=0.1, s=10)
plt.plot([0, max(true_mags)], [0, max(true_mags)], 'r--')
plt.xlabel("True Spatial Magnitude")
plt.ylabel("Predicted Spatial Magnitude")
plt.title("Parity Plot: Actual vs Predicted")
plt.savefig(os.path.join(CACHE_DIR, "parity_plot.png"))
plt.show()

# --- 8. FINAL EXTRACTION & SAVING ---
print("\n📦 Extracting Final Embeddings (Best Model)...")

# Load Best Weights
semantic_vae.load_state_dict(torch.load(os.path.join(MODEL_DIR, "best_semantic_vae.pth")))
semantic_vae.eval()

embeddings = []
# Use full dataset for final extraction
full_loader = DataLoader(full_dataset, batch_size=32, shuffle=False)

with torch.no_grad():
    for batch in tqdm(full_loader, desc="Encoding All Cells"):
        batch = batch.to(DEVICE)
        h_pooled = semantic_vae.pool(batch.x, batch.batch)
        mu = semantic_vae.fc_mu(h_pooled)
        embeddings.append(mu.cpu().numpy())

final_z = np.vstack(embeddings)
adata_semantic = sc.AnnData(final_z)

if len(adata_semantic) <= len(adata):
    adata_semantic.obs = adata.obs.iloc[:len(adata_semantic)].copy()

sc.pp.neighbors(adata_semantic)
sc.tl.umap(adata_semantic)

sc.pl.umap(
    adata_semantic, 
    color=['condition', 'cell_type'] if 'condition' in adata_semantic.obs else None,
    title='Stable Semantic Spatial State'
)

save_path = os.path.join(CACHE_DIR, "semantic_spatial_atlas.h5ad")
adata_semantic.write(save_path)
print(f"💾 Saved Final Atlas to: {save_path}")

In [ ]:
adata_semantic

In [ ]:
adata.obs.cell_group.value_counts()

# SUPERVISED VAE

In [ ]:
model

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.data import Dataset, Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GlobalAttention
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import numpy as np
import pandas as pd
import scanpy as sc
import os
from tqdm import tqdm

# --- 1. CONFIGURATION ---
BATCH_SIZE = 32
INPUT_DIM = 16        
HIDDEN_DIM = 64       
EPOCHS = 35
LR = 1e-4
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CACHE_DIR = "/home/projects/nyosef/zvise/PixelGen/PixelGen/cache/GVAE/multi_task_model/attention_nodes"
os.makedirs(CACHE_DIR, exist_ok=True)

print("🚀 Starting Multi-Task Attention Pipeline")

# --- 2. PREPARE DUAL LABELS ---
print("   • Preparing dual labels (Cell Type + Condition)...")
valid_files = []
labels_type = [] # e.g., 'CD4', 'B_cell'
labels_cond = [] # e.g., '4hrs', 'control'

# Create mappings
id_to_obs = adata.obs[['cell_group', 'condition']].to_dict('index')

for f in files:
    cell_id = os.path.basename(f).replace('.pt', '') 
    if cell_id in id_to_obs:
        valid_files.append(f)
        labels_type.append(id_to_obs[cell_id]['cell_group'])
        labels_cond.append(id_to_obs[cell_id]['condition'])

# Encode Both
le_type = LabelEncoder()
y_type_encoded = le_type.fit_transform(labels_type)

le_cond = LabelEncoder()
y_cond_encoded = le_cond.fit_transform(labels_cond)

num_classes_type = len(le_type.classes_)
num_classes_cond = len(le_cond.classes_)

print(f"   • {len(valid_files)} cells")
print(f"   • Task 1: {num_classes_type} Cell Types")
print(f"   • Task 2: {num_classes_cond} Conditions")

# Split (Stratify by Condition to ensure balance on the hard task)
split_data = train_test_split(
    valid_files, y_type_encoded, y_cond_encoded, 
    test_size=0.2, random_state=42, stratify=y_cond_encoded
)
files_train, files_test, y_type_train, y_type_test, y_cond_train, y_cond_test = split_data

# --- 3. DUAL-LABEL DATASET ---
class MultiTaskDataset(Dataset):
    def __init__(self, file_list, y_type, y_cond, gvae_model, device):
        super().__init__()
        self.file_list = file_list
        self.y_type = y_type
        self.y_cond = y_cond
        self.gvae_model = gvae_model
        self.device = device
        self.gvae_model.eval().to(device)

    def len(self):
        return len(self.file_list)

    def get(self, idx):
        path = self.file_list[idx]
        data = torch.load(path, weights_only=False).to(self.device)
        
        with torch.no_grad():
            z_nodes = self.gvae_model.encode(data.x, data.edge_index)
            
        return Data(
            x=z_nodes.cpu(), 
            y_type=torch.tensor(self.y_type[idx], dtype=torch.long),
            y_cond=torch.tensor(self.y_cond[idx], dtype=torch.long),
            num_nodes=data.num_nodes
        )

# --- 4. THE MULTI-HEAD MODEL ---
class MultiHeadClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_types, num_conds):
        super(MultiHeadClassifier, self).__init__()
        
        # Shared Attention Backbone
        self.att_gate = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.Tanh(),
            nn.Linear(32, 1)
        )
        self.pool = GlobalAttention(gate_nn=self.att_gate)
        
        # Shared Embedding Layer
        self.shared_fc = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        
        # HEAD 1: Cell Type (Easy Task)
        self.head_type = nn.Linear(hidden_dim, num_types)
        
        # HEAD 2: Condition (Hard Task)
        self.head_cond = nn.Linear(hidden_dim, num_conds)

    def forward(self, x, batch):
        # 1. Shared Feature Extraction
        h_pooled = self.pool(x, batch)
        embedding = self.shared_fc(h_pooled)
        
        # 2. Multi-Head Prediction
        out_type = self.head_type(embedding)
        out_cond = self.head_cond(embedding)
        
        return out_type, out_cond, embedding

# --- 5. TRAINING LOOP ---
train_ds = MultiTaskDataset(files_train, y_type_train, y_cond_train, model, DEVICE)
test_ds = MultiTaskDataset(files_test, y_type_test, y_cond_test, model, DEVICE)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

classifier = MultiHeadClassifier(INPUT_DIM, HIDDEN_DIM, num_classes_type, num_classes_cond).to(DEVICE)
optimizer = optim.Adam(classifier.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

print("\n🔄 Training Multi-Head Model...")

for epoch in range(EPOCHS):
    classifier.train()
    total_loss = 0
    acc_type = 0
    acc_cond = 0
    total = 0
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=True)
    
    for batch in loop:
        batch = batch.to(DEVICE)
        optimizer.zero_grad()
        
        # Forward
        pred_type, pred_cond, _ = classifier(batch.x, batch.batch)
        
        # Calculate Separate Losses
        loss_type = criterion(pred_type, batch.y_type)
        loss_cond = criterion(pred_cond, batch.y_cond)
        
        # Combine Losses 
        # (We can weight them: 1.0 for type, 1.0 for condition)
        loss = loss_type + loss_cond
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        # Accuracy Tracking
        acc_type += (pred_type.argmax(1) == batch.y_type).sum().item()
        acc_cond += (pred_cond.argmax(1) == batch.y_cond).sum().item()
        total += batch.y_cond.size(0)
        
        loop.set_postfix(
            loss=loss.item(), 
            type_acc=acc_type/total, 
            cond_acc=acc_cond/total
        )

# --- 6. EXTRACT & PLOT ---
print("\n📦 Extracting Multi-Task Embeddings...")
full_ds = MultiTaskDataset(valid_files, y_type_encoded, y_cond_encoded, model, DEVICE)
full_loader = DataLoader(full_ds, batch_size=32, shuffle=False)

classifier.eval()
embeddings = []
l_type = []
l_cond = []

with torch.no_grad():
    for batch in tqdm(full_loader):
        batch = batch.to(DEVICE)
        _, _, emb = classifier(batch.x, batch.batch)
        
        embeddings.append(emb.cpu().numpy())
        l_type.extend(batch.y_type.cpu().numpy())
        l_cond.extend(batch.y_cond.cpu().numpy())

final_z = np.vstack(embeddings)

adata_mt = sc.AnnData(final_z)
adata_mt.obs['pred_cell_type'] = le_type.inverse_transform(l_type)
adata_mt.obs['pred_condition'] = le_cond.inverse_transform(l_cond)
# Add original metadata for comparison if needed
# adata_mt.obs['true_condition'] = ... 

print("🎨 Running UMAP...")
sc.pp.neighbors(adata_mt, n_neighbors=30)
sc.tl.umap(adata_mt)

# Plot both views side-by-side
sc.pl.umap(
    adata_mt, 
    color=['pred_cell_type', 'pred_condition'], 
    title=['Learned Structure (Cell Type)', 'Learned Structure (Condition)'],
    wspace=0.3
)

adata_mt.write(os.path.join(CACHE_DIR, "multitask_spatial_atlas.h5ad"))

In [ ]:
model

In [ ]:
final_z.shape

In [ ]:
np.save("/home/projects/nyosef/zvise/PixelGen/PixelGen/cache/GVAE/attention_embeddings.npy", final_z)


# multi head attention model

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.data import Dataset, Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GlobalAttention, GATv2Conv, VGAE
from torch_geometric.utils import softmax
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
import glob
from tqdm import tqdm
import gc

# --- 1. CONFIGURATION ---
BATCH_SIZE = 8          # Low batch size to prevent OOM
INPUT_DIM = 16          # Latent dim from GVAE
HIDDEN_DIM = 64         # Classifier hidden dim
NUM_HEADS = 4           # Number of biological patterns to learn
EPOCHS = 50
LR = 1e-4
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Paths (Adjust if needed)
BASE_DIR = "/home/projects/nyosef/zvise/PixelGen/PixelGen/cache/GVAE"
GVAE_PATH = os.path.join(BASE_DIR, "models/gvae_gat_model.pth")
CACHE_DIR = os.path.join(BASE_DIR, "supervised_attention")
OUTPUT_DIR = "/home/projects/nyosef/zvise/PixelGen/PixelGen/cache/GVAE/graphs" # Where your .pt files are
os.makedirs(CACHE_DIR, exist_ok=True)

print(f"🚀 Starting Supervised Attention Pipeline on {DEVICE}")

# --- 2. MODEL DEFINITIONS ---

# A. The Micro-Model (Frozen GVAE) - Must match training exactly
class VariationalGATEncoder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        # Heads=4, concat=True -> Output size is hidden_channels * 4
        self.conv1 = GATv2Conv(in_channels, hidden_channels, heads=4, concat=True)
        hidden_out = hidden_channels * 4
        # Heads=1, concat=False -> Output size is out_channels
        self.conv_mu = GATv2Conv(hidden_out, out_channels, heads=1, concat=False)
        self.conv_logstd = GATv2Conv(hidden_out, out_channels, heads=1, concat=False)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        return self.conv_mu(x, edge_index), self.conv_logstd(x, edge_index)

# B. The Macro-Model (Multi-Head Classifier)
class MultiHeadAttentionClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        
        # 1. Independent Attention Gates
        self.attention_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(input_dim, 32),
                nn.Tanh(),
                nn.Linear(32, 1)
            ) for _ in range(num_heads)
        ])
        
        # 2. Pooling Layers
        self.poolers = nn.ModuleList([
            GlobalAttention(gate_nn=gate) for gate in self.attention_heads
        ])
        
        # 3. Classifier
        # Input is (input_dim * num_heads) because we concat the results
        self.classifier = nn.Sequential(
            nn.Linear(input_dim * num_heads, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x, batch):
        pooled_outputs = []
        for pooler in self.poolers:
            h = pooler(x, batch) 
            pooled_outputs.append(h)
            
        h_concat = torch.cat(pooled_outputs, dim=1)
        logits = self.classifier(h_concat)
        return logits, h_concat

    def get_attention_weights(self, x, batch):
        """Returns list of attention weights (one per head)"""
        weights = []
        for gate in self.attention_heads:
            x_gate = gate(x) 
            alpha = softmax(x_gate, batch)
            weights.append(alpha.detach().cpu().numpy())
        return weights

# --- 3. DATASET & LOADING ---
class LabeledGraphDataset(Dataset):
    def __init__(self, file_list, labels, gvae_model, device):
        super().__init__()
        self.file_list = file_list
        self.labels = labels
        self.gvae_model = gvae_model
        self.device = device
        
        # Freeze GVAE to save memory
        self.gvae_model.eval()
        for param in self.gvae_model.parameters():
            param.requires_grad = False

    def len(self):
        return len(self.file_list)

    def get(self, idx):
        path = self.file_list[idx]
        data = torch.load(path, weights_only=False).to(self.device)
        
        # Transform Raw Graph -> Embeddings
        with torch.no_grad():
            z_nodes = self.gvae_model.encode(data.x, data.edge_index)
        
        # Return embeddings, label, and raw protein IDs
        return Data(
            x=z_nodes.cpu(), 
            y=torch.tensor(self.labels[idx], dtype=torch.long),
            protein_id=data.x.argmax(dim=1).cpu(),
            num_nodes=data.num_nodes
        )

# --- 4. EXECUTION PIPELINE ---

# A. Load Frozen GVAE
print("❄️ Loading Frozen GVAE...")
gvae_encoder = VariationalGATEncoder(159, 64, 16) 
gvae_model = VGAE(gvae_encoder).to(DEVICE)
gvae_model.load_state_dict(torch.load(GVAE_PATH, map_location=DEVICE, weights_only=True))
gvae_model.eval()

# B. Prepare Labels (Control / 4hrs / 24hrs)
print("🏷️ Preparing Data...")
files = glob.glob(f"{OUTPUT_DIR}/*.pt")
valid_files = []
valid_labels = []

# Ensure 'adata' is available in your environment!
id_to_cond = adata.obs['condition'].to_dict() 

for f in files:
    cell_id = os.path.basename(f).replace('.pt', '')
    if cell_id in id_to_cond:
        valid_files.append(f)
        valid_labels.append(id_to_cond[cell_id])

le = LabelEncoder()
y_encoded = le.fit_transform(valid_labels)
classes = le.classes_
print(f"   • Found {len(valid_files)} cells")
print(f"   • Conditions: {dict(zip(classes, range(len(classes))))}")

X_train, X_test, y_train, y_test = train_test_split(
    valid_files, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42
)

train_ds = LabeledGraphDataset(X_train, y_train, gvae_model, DEVICE)
test_ds = LabeledGraphDataset(X_test, y_test, gvae_model, DEVICE)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

# C. Train Classifier
model = MultiHeadAttentionClassifier(INPUT_DIM, HIDDEN_DIM, len(classes), NUM_HEADS).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
criterion = nn.CrossEntropyLoss()

print("\n🔄 Training Classifier...")
best_val_acc = 0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=True)
    
    for batch in loop:
        batch = batch.to(DEVICE)
        optimizer.zero_grad()
        
        logits, _ = model(batch.x, batch.batch)
        loss = criterion(logits, batch.y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        preds = logits.argmax(dim=1)
        correct += (preds == batch.y).sum().item()
        total += batch.y.size(0)
        loop.set_postfix(loss=loss.item(), acc=correct/total)
    
    # Validation
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(DEVICE)
            logits, _ = model(batch.x, batch.batch)
            preds = logits.argmax(dim=1)
            val_correct += (preds == batch.y).sum().item()
            val_total += batch.y.size(0)
    
    val_acc = val_correct/val_total
    print(f"   Train Acc: {correct/total:.2f} | Val Acc: {val_acc:.2f}")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), os.path.join(CACHE_DIR, "best_attention_classifier.pth"))

# --- 5. INTERPRETATION & PLOTS ---

def analyze_attention(model, loader, marker_map, target_class_idx):
    """Finds top proteins driving the decision for a specific class"""
    model.eval()
    protein_scores = {p: [] for p in marker_map.keys()}
    idx_to_marker = {v: k for k, v in marker_map.items()}
    
    print(f"\n🕵️‍♀️ Analyzing Attention for Class Index {target_class_idx}...")
    
    with torch.no_grad():
        for batch in tqdm(loader, desc="Scanning"):
            batch = batch.to(DEVICE)
            logits, _ = model(batch.x, batch.batch)
            preds = logits.argmax(dim=1)
            
            # Mask for correctly predicted graphs of target class
            target_mask = (preds == target_class_idx) & (batch.y == target_class_idx)
            if not target_mask.any(): continue
                
            weights_list = model.get_attention_weights(batch.x, batch.batch)
            avg_weights = np.mean(np.array(weights_list), axis=0).flatten()
            
            batch_np = batch.batch.cpu().numpy()
            target_indices = np.where(target_mask.cpu().numpy())[0]
            node_mask = np.isin(batch_np, target_indices)
            
            relevant_weights = avg_weights[node_mask]
            relevant_ids = batch.protein_id.cpu().numpy()[node_mask]
            
            for pid, w in zip(relevant_ids, relevant_weights):
                p_name = idx_to_marker.get(pid, "Unknown")
                protein_scores[p_name].append(w)
    
    results = []
    for p, scores in protein_scores.items():
        if len(scores) > 0:
            results.append({
                "Protein": p, 
                "Mean_Att": np.mean(scores),
                "Max_Att": np.max(scores), 
                "Count": len(scores)
            })
    return pd.DataFrame(results).sort_values("Mean_Att", ascending=False)

# Load best model
model.load_state_dict(torch.load(os.path.join(CACHE_DIR, "best_attention_classifier.pth")))

# Plot Confusion Matrix
print("\n📊 Generating Confusion Matrix...")
model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(DEVICE)
        logits, _ = model(batch.x, batch.batch)
        all_preds.extend(logits.argmax(dim=1).cpu().numpy())
        all_labels.extend(batch.y.cpu().numpy())

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title('Prediction Accuracy by Condition')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

# Analyze Top Proteins for "24h" (Assuming index 1, change based on print output!)


In [ ]:
def analyze_attention(model, loader, marker_map, target_class_idx):
    """Finds top proteins driving the decision for a specific class"""
    model.eval()
    protein_scores = {p: [] for p in marker_map.keys()}
    idx_to_marker = {v: k for k, v in marker_map.items()}
    
    print(f"\n🕵️‍♀️ Analyzing Attention for Class Index {target_class_idx}...")
    
    with torch.no_grad():
        for batch in tqdm(loader, desc="Scanning"):
            batch = batch.to(DEVICE)
            logits, _ = model(batch.x, batch.batch)
            preds = logits.argmax(dim=1)
            
            # Mask for correctly predicted graphs of target class
            target_mask = (preds == target_class_idx) & (batch.y == target_class_idx)
            if not target_mask.any(): continue
                
            weights_list = model.get_attention_weights(batch.x, batch.batch)
            avg_weights = np.mean(np.array(weights_list), axis=0).flatten()
            
            batch_np = batch.batch.cpu().numpy()
            target_indices = np.where(target_mask.cpu().numpy())[0]
            node_mask = np.isin(batch_np, target_indices)
            
            relevant_weights = avg_weights[node_mask]
            relevant_ids = batch.protein_id.cpu().numpy()[node_mask]
            
            for pid, w in zip(relevant_ids, relevant_weights):
                p_name = idx_to_marker.get(pid, "Unknown")
                protein_scores[p_name].append(w)
    
    results = []
    for p, scores in protein_scores.items():
        if len(scores) > 0:
            results.append({
                "Protein": p, 
                "Mean_Att": np.mean(scores),
                "Max_Att": np.max(scores), 
                "Count": len(scores)
            })
    return pd.DataFrame(results).sort_values("Mean_Att", ascending=False)

# Load best model
model.load_state_dict(torch.load(os.path.join(CACHE_DIR, "best_attention_classifier.pth")))

In [ ]:
print("\n📊 Generating Confusion Matrix...")
model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(DEVICE)
        logits, _ = model(batch.x, batch.batch)
        all_preds.extend(logits.argmax(dim=1).cpu().numpy())
        all_labels.extend(batch.y.cpu().numpy())

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title('Prediction Accuracy by Condition')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

In [ ]:
target_idx = 1 
df_res = analyze_attention(model, test_loader, MARKER_TO_IDX, target_idx)
print(df_res.head(10))

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def plot_supervised_umap(model, loader, class_names):
    """
    Extracts the 'Deep Attention State' from the classifier and plots UMAP.
    """
    print("🎨 Extracting Supervised Embeddings...")
    model.eval()
    
    embeddings = []
    labels = []
    predictions = []
    
    with torch.no_grad():
        for batch in tqdm(loader, desc="Encoding Cells"):
            batch = batch.to(DEVICE)
            
            # 1. Forward pass to get the 'h_concat' vector
            # (logits, h_concat) = model(x, batch)
            logits, h_concat = model(batch.x, batch.batch)
            
            # 2. Store the embedding (The "Concept" of the cell)
            embeddings.append(h_concat.cpu().numpy())
            
            # 3. Store Metadata
            labels.extend(batch.y.cpu().numpy())
            predictions.extend(logits.argmax(dim=1).cpu().numpy())

    # Stack into a big matrix [Num_Cells, Hidden_Dim * Num_Heads]
    X_latents = np.vstack(embeddings)
    
    # --- Create AnnData for Scanpy ---
    adata_sup = sc.AnnData(X=X_latents)
    
    # Add metadata
    # Map integer labels back to string names (e.g., 0 -> '4_hrs')
    true_names = [class_names[i] for i in labels]
    pred_names = [class_names[i] for i in predictions]
    
    adata_sup.obs['Condition'] = true_names
    adata_sup.obs['Predicted_Condition'] = pred_names
    adata_sup.obs['Is_Correct'] = (np.array(labels) == np.array(predictions)).astype(str)

    # --- Run UMAP ---
    print("   • Computing Neighbors...")
    sc.pp.neighbors(adata_sup, n_neighbors=15, use_rep='X') # Use the latents directly
    
    print("   • Running UMAP...")
    sc.tl.umap(adata_sup)
    
    # --- Plot ---
    print("   • Plotting...")
    sc.pl.umap(
        adata_sup, 
        color=['Condition', 'Predicted_Condition'], 
        title=['True Biological Condition', 'Model Prediction'],
        wspace=0.3
    )
    
    return adata_sup

# --- EXECUTE ---
# Use the full loader or test loader
# Note: 'classes' comes from the LabelEncoder in the previous script
adata_supervised = plot_supervised_umap(model, test_loader, classes)

# Optional: Save this new Atlas
# adata_supervised.write(os.path.join(CACHE_DIR, "supervised_attention_atlas.h5ad"))

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import torch
import torch.nn.functional as F

def get_latent_embeddings(model, loader, class_names):
    """
    Extracts the 'Deep Attention State' (z) for every cell.
    Returns: DataFrame with embeddings + metadata.
    """
    print("📦 Extracting Z-Embeddings...")
    model.eval()
    
    embeddings = []
    meta_data = []
    
    with torch.no_grad():
        for batch in tqdm(loader, desc="Encoding"):
            batch = batch.to(DEVICE)
            
            # 1. Forward Pass
            # h_concat is the latent vector [Batch, Hidden_Dim * Num_Heads]
            logits, h_concat = model(batch.x, batch.batch)
            
            # 2. Get Predictions & Confidence
            probs = F.softmax(logits, dim=1)
            pred_conf, pred_idx = probs.max(dim=1)
            
            # 3. Store Data
            # Convert tensors to numpy
            z_vecs = h_concat.cpu().numpy()
            y_true = batch.y.cpu().numpy()
            y_pred = pred_idx.cpu().numpy()
            conf = pred_conf.cpu().numpy()
            
            # Append rows
            embeddings.append(z_vecs)
            
            # Store metadata for these cells
            for i in range(len(y_true)):
                meta_data.append({
                    "Condition": class_names[y_true[i]],
                    "Predicted": class_names[y_pred[i]],
                    "Confidence": conf[i],
                    "Is_Correct": y_true[i] == y_pred[i]
                })

    # --- FORMATTING ---
    # 1. Create the Z matrix
    X_z = np.vstack(embeddings)
    
    # 2. Create the Metadata table
    df_meta = pd.DataFrame(meta_data)
    
    # 3. Create the Embeddings table
    # Columns named z_0, z_1, ... z_255
    col_names = [f"z_{i}" for i in range(X_z.shape[1])]
    df_z = pd.DataFrame(X_z, columns=col_names)
    
    # 4. Merge
    df_final = pd.concat([df_meta, df_z], axis=1)
    
    print(f"✅ Extracted embeddings for {len(df_final)} cells.")
    print(f"   • Latent Shape: {X_z.shape}")
    
    return df_final

# --- EXECUTE ---
# Use the full dataset loader if you want ALL cells, or test_loader for just test
df_latents = get_latent_embeddings(model, test_loader, classes)

# Preview
print(df_latents.head())

# Save to CSV if needed for external plotting
# df_latents.to_csv(os.path.join(CACHE_DIR, "supervised_latents.csv"), index=False)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GlobalAttention, GATv2Conv, VGAE
from torch_geometric.loader import DataLoader
from torch_geometric.data import Dataset, Data
import pandas as pd
import numpy as np
from tqdm import tqdm
import os
import glob
import scanpy as sc

# --- 1. SETTINGS ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 8
# Update these paths to where your models actually are
GVAE_PATH = "/home/projects/nyosef/zvise/PixelGen/PixelGen/cache/GVAE/models/gvae_gat_model.pth"
CLASSIFIER_PATH = "/home/projects/nyosef/zvise/PixelGen/PixelGen/cache/GVAE/supervised_attention/best_attention_classifier.pth"
DATA_DIR = "/home/projects/nyosef/zvise/PixelGen/PixelGen/cache/GVAE/graphs" # Folder containing your .pt files

# --- 2. DEFINE MODELS (Must match training) ---
class VariationalGATEncoder(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GATv2Conv(in_channels, hidden_channels, heads=4, concat=True)
        hidden_out = hidden_channels * 4
        self.conv_mu = GATv2Conv(hidden_out, out_channels, heads=1, concat=False)
        self.conv_logstd = GATv2Conv(hidden_out, out_channels, heads=1, concat=False)
    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        return self.conv_mu(x, edge_index), self.conv_logstd(x, edge_index)

class MultiHeadAttentionClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes, num_heads=4):
        super().__init__()
        self.attention_heads = nn.ModuleList([
            nn.Sequential(nn.Linear(input_dim, 32), nn.Tanh(), nn.Linear(32, 1)) 
            for _ in range(num_heads)
        ])
        self.poolers = nn.ModuleList([GlobalAttention(gate_nn=gate) for gate in self.attention_heads])
        self.classifier = nn.Sequential(
            nn.Linear(input_dim * num_heads, hidden_dim),
            nn.BatchNorm1d(hidden_dim), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(hidden_dim, num_classes)
        )
    def forward(self, x, batch):
        pooled = [pooler(x, batch) for pooler in self.poolers]
        h_concat = torch.cat(pooled, dim=1)
        return self.classifier(h_concat), h_concat

# --- 3. DATA LOADER ---
class SimpleGraphDataset(Dataset):
    def __init__(self, file_list, gvae_model):
        super().__init__()
        self.file_list = file_list
        self.gvae_model = gvae_model
        self.gvae_model.eval()

    def len(self):
        return len(self.file_list)

    def get(self, idx):
        path = self.file_list[idx]
        data = torch.load(path, weights_only=False).to(DEVICE)
        # Convert to Node Embeddings immediately
        with torch.no_grad():
            z_nodes = self.gvae_model.encode(data.x, data.edge_index)
        return Data(x=z_nodes.cpu(), num_nodes=data.num_nodes, path=path)

# --- 4. EXECUTION ---
print("🚀 Loading Models...")

# Load GVAE
gvae = VGAE(VariationalGATEncoder(159, 64, 16)).to(DEVICE)
gvae.load_state_dict(torch.load(GVAE_PATH, map_location=DEVICE, weights_only=True))

# Load Classifier
# Note: num_classes=3 for (Control, 4h, 24h). Adjust if you have a different number.
classifier = MultiHeadAttentionClassifier(16, 64, num_classes=3, num_heads=4).to(DEVICE)
classifier.load_state_dict(torch.load(CLASSIFIER_PATH, map_location=DEVICE, weights_only=True))
classifier.eval()

# Prepare Data
all_files = glob.glob(f"{DATA_DIR}/*.pt")
ds = SimpleGraphDataset(all_files, gvae)
loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"📦 Extracting embeddings for {len(all_files)} cells...")

embeddings = []
filenames = []

with torch.no_grad():
    for batch in tqdm(loader):
        batch = batch.to(DEVICE)
        
        # Get Cell Embedding (h_concat)
        _, h_concat = classifier(batch.x, batch.batch)
        
        embeddings.append(h_concat.cpu().numpy())
        filenames.extend(batch.path)

# --- 5. SAVE ---
# Create simple DataFrame
X = np.vstack(embeddings)
df = pd.DataFrame(X, columns=[f"z_{i}" for i in range(X.shape[1])])
df.index = [os.path.basename(f).replace('.pt', '') for f in filenames]

print(f"✅ Done. Shape: {df.shape}")
print(df.head())

# Save
# df.to_csv("all_cell_embeddings.csv")

In [ ]:
classifier

In [ ]:
adata.obsm['X_supervised_attention'] = df.loc[adata.obs_names].values
# sc.pp.pca(adata_spatdaial)
sc.pp.neighbors(adata, use_rep='X_supervised_attention')
sc.tl.umap(adata)

# Plot
sc.pl.umap(
    adata, 
    color=['condition', 'cell_group','cell_type'], 
    title=['Spatial State (Condition)', 'Spatial State (Cell Type)'], 
    wspace=0.3
)

In [ ]:
layout=data.filter(components=adata.obs_names[0]).precomputed_layouts().to_df()

In [ ]:
import plotly.express as px
import numpy as np
import torch
import os
from torch.utils.data import Dataset 
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

# --- 1. DATASET WRAPPER (Same as before) ---
class VizDataset(Dataset):
    def __init__(self, file_list, labels, gvae_model):
        self.file_list = file_list
        self.labels = labels
        self.gvae_model = gvae_model
        self.gvae_model.eval()

    def __len__(self): return len(self.file_list)

    def __getitem__(self, idx): 
        path = self.file_list[idx]
        d = torch.load(path, weights_only=False).to(DEVICE)
        with torch.no_grad(): z = self.gvae_model.encode(d.x, d.edge_index)
        return Data(x=z.cpu(), y=torch.tensor(self.labels[idx]), path=path, batch=torch.zeros(d.num_nodes, dtype=torch.long))

# --- 2. VISUALIZATION FUNCTION (Smaller Dots) ---
def visualize_archetype(model, loader, layout_data, target_class=1):
    print(f"🔎 Scanning for high-confidence Class {target_class} (24h) cell...")
    model.eval()
    
    best = {"conf": 0, "id": None, "att": None}
    
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(DEVICE)
            logits, _ = model(batch.x, batch.batch)
            probs = torch.nn.functional.softmax(logits, dim=1)
            
            conf, pred = probs.max(dim=1)
            
            if pred == target_class and batch.y == target_class and conf > best["conf"]:
                weights = model.get_attention_weights(batch.x, batch.batch)
                best["conf"] = conf.item()
                best["id"] = os.path.basename(batch.path[0]).replace('.pt', '')
                best["att"] = np.mean(np.array(weights), axis=0).flatten()

    if not best["id"]: return print("❌ No matching cell found.")

    print(f"🏆 Found Archetype: {best['id']} (Conf: {best['conf']:.4f})")
    
    # --- MERGE & LOG-TRANSFORM ---
    df = layout_data.filter(components=best["id"]).precomputed_layouts().to_df()
    
    min_len = min(len(df), len(best["att"]))
    df = df.iloc[:min_len].copy()
    raw_att = best["att"][:min_len]

    # Log scaling for better visualization
    scaled_att = raw_att / np.mean(raw_att)
    df['Log_Attention'] = np.log(scaled_att + 1e-5)
    
    print("   🎨 Generating Log-Scaled Plot...")
    fig = px.scatter_3d(
        df, x='x', y='y', z='z', 
        color='Log_Attention',
        hover_data=['pixel_type'], 
        color_continuous_scale='Magma', 
        title=f"Log-Attention Map: {best['id']} (24h)", 
        opacity=0.6 # Lowered opacity for dense clouds
    )
    
    # --- THE FIX: MAKE DOTS SMALLER ---
    # Set fixed size for all markers. Adjust '3' up or down as needed.
    fig.update_traces(marker=dict(size=3)) 
    
    fig.update_layout(scene=dict(xaxis=dict(visible=False), yaxis=dict(visible=False), zaxis=dict(visible=False)))
    fig.write_html(
        '/home/projects/nyosef/zvise/PixelGen/PixelGen/figures',
        include_plotlyjs="cdn",
        full_html=True
    )
    print(f"   💾 Saved interactive HTML to: {html_path}")
    fig.show()

# --- EXECUTION ---
viz_ds = VizDataset(X_test, y_test, gvae_model)
viz_loader = DataLoader(viz_ds, batch_size=1, shuffle=False)

visualize_archetype(model, viz_loader, data, target_class=1)

# SECOND UNSUPERVISED

## TRAIN THE MODEL

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.data import Dataset, Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GlobalAttention, GATv2Conv, VGAE
from torch_geometric.utils import scatter
from sklearn.model_selection import train_test_split
import numpy as np
import os
import glob
from tqdm import tqdm

# --- 1. CONFIGURATION ---
BATCH_SIZE = 8          
INPUT_DIM = 16          # Latent dim from Frozen GVAE
NUM_PROTEINS = 159      # Total protein types in dataset
PROTEIN_EMB_DIM = 16    # Dimension to encode protein identity
HIDDEN_DIM = 128        # Internal VAE width
LATENT_DIM = 32         # The "Cell State" Embedding
EPOCHS = 100            # Increased for "Texture Learning"
PATIENCE = 15           # Early stopping patience
LR = 1e-4
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Paths
BASE_DIR = "/home/projects/nyosef/zvise/PixelGen/PixelGen/cache/GVAE"
GVAE_PATH = os.path.join(BASE_DIR, "models/gvae_gat_model.pth")
OUTPUT_DIR = "/home/projects/nyosef/zvise/PixelGen/PixelGen/cache/GVAE/graphs"
CACHE_DIR = os.path.join(BASE_DIR, "moment_matching_vae")
os.makedirs(CACHE_DIR, exist_ok=True)

print(f"🚀 Starting Moment-Matching VAE on {DEVICE}")

# --- 2. MODEL DEFINITIONS ---

# A. The Micro-Model (Frozen GVAE)
class VariationalGATEncoder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GATv2Conv(in_channels, hidden_channels, heads=4, concat=True)
        hidden_out = hidden_channels * 4
        self.conv_mu = GATv2Conv(hidden_out, out_channels, heads=1, concat=False)
        self.conv_logstd = GATv2Conv(hidden_out, out_channels, heads=1, concat=False)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        return self.conv_mu(x, edge_index), self.conv_logstd(x, edge_index)

# B. The Macro-Model (Moment Matching VAE)
class MomentMatchingVAE(nn.Module):
    def __init__(self):
        super().__init__()
        
        # --- ENCODER ---
        self.prot_encoder = nn.Linear(NUM_PROTEINS, PROTEIN_EMB_DIM)
        
        # Input = Spatial (16) + Protein (16) = 32
        fusion_dim = INPUT_DIM + PROTEIN_EMB_DIM
        
        # Attention Pooling ("Which protein-locations matter?")
        self.att_gate = nn.Sequential(
            nn.Linear(fusion_dim, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )
        self.pool = GlobalAttention(gate_nn=self.att_gate)
        
        # Encoder Core
        self.enc_fc = nn.Sequential(
            nn.Linear(fusion_dim, HIDDEN_DIM),
            nn.ReLU(),
            nn.BatchNorm1d(HIDDEN_DIM)
        )
        
        # Variational Heads
        self.fc_mu = nn.Linear(HIDDEN_DIM, LATENT_DIM)
        self.fc_var = nn.Linear(HIDDEN_DIM, LATENT_DIM)
        
        # --- DECODERS (FULL COVARIANCE) ---
        
        # 1. Mean Head (Location) -> [159 proteins * 16 dims]
        self.decoder_mu = nn.Sequential(
            nn.Linear(LATENT_DIM, HIDDEN_DIM),
            nn.ReLU(),
            nn.Linear(HIDDEN_DIM, NUM_PROTEINS * INPUT_DIM)
        )
        
        # 2. Variance Head (Shape) -> [159 proteins * 16 dims]
        self.decoder_var = nn.Sequential(
            nn.Linear(LATENT_DIM, HIDDEN_DIM),
            nn.ReLU(),
            nn.Linear(HIDDEN_DIM, NUM_PROTEINS * INPUT_DIM),
            nn.Softplus() # Variance must be positive
        )

    def encode(self, z_spatial, x_onehot, batch):
        # Embed Protein Identity
        h_prot = F.relu(self.prot_encoder(x_onehot))
        # Fuse Spatial + Identity
        h_fused = torch.cat([z_spatial, h_prot], dim=1)
        # Pool
        graph_h = self.pool(h_fused, batch)
        # Latent
        h = self.enc_fc(graph_h)
        return self.fc_mu(h), self.fc_var(h)

    def reparameterize(self, mu, logstd):
        if self.training:
            return mu + torch.randn_like(mu) * torch.exp(logstd)
        return mu

    def forward(self, z_spatial, x_onehot, batch):
        mu, logstd = self.encode(z_spatial, x_onehot, batch)
        z_cell = self.reparameterize(mu, logstd)
        
        # Decode & Reshape: [Batch, Proteins, Dims]
        pred_means = self.decoder_mu(z_cell).view(-1, NUM_PROTEINS, INPUT_DIM)
        pred_vars = self.decoder_var(z_cell).view(-1, NUM_PROTEINS, INPUT_DIM)
        
        return z_cell, pred_means, pred_vars, mu, logstd

# --- 3. LOSS FUNCTION ---
def moment_loss(pred_means, pred_vars, z_spatial, x_onehot, batch, mu, logstd):
    # 1. Setup Indices
    prot_ids = x_onehot.argmax(dim=1)
    batch_size = batch.max().item() + 1
    unique_indices = batch * NUM_PROTEINS + prot_ids
    dim_size = batch_size * NUM_PROTEINS
    
    # 2. Calculate Ground Truth Empirical Moments
    # Mean (Location)
    true_means_flat = scatter(z_spatial, unique_indices, dim=0, reduce='mean', dim_size=dim_size)
    
    # Variance (Shape) = E[X^2] - E[X]^2
    true_sq_means_flat = scatter(z_spatial.pow(2), unique_indices, dim=0, reduce='mean', dim_size=dim_size)
    true_vars_flat = torch.clamp(true_sq_means_flat - true_means_flat.pow(2), min=1e-6)
    
    # Reshape
    true_means = true_means_flat.view(batch_size, NUM_PROTEINS, INPUT_DIM)
    true_vars = true_vars_flat.view(batch_size, NUM_PROTEINS, INPUT_DIM)
    
    # 3. Masking (Ignore proteins not present in the graph)
    counts_flat = scatter(torch.ones_like(prot_ids), unique_indices, dim=0, reduce='sum', dim_size=dim_size)
    counts = counts_flat.view(batch_size, NUM_PROTEINS)
    # We need at least 2 nodes to calculate a meaningful variance
    mask = (counts > 1).unsqueeze(-1).expand(-1, -1, INPUT_DIM)
    
    # 4. Compute Loss
    if mask.sum() > 0:
        loss_mu = F.mse_loss(pred_means[mask], true_means[mask])
        loss_var = F.mse_loss(torch.log(pred_vars[mask] + 1e-9), torch.log(true_vars[mask] + 1e-9))
    else:
        loss_mu, loss_var = 0.0, 0.0

    # KL Divergence
    kl_loss = -0.5 * torch.mean(torch.sum(1 + 2 * logstd - mu**2 - torch.exp(2 * logstd), dim=1))
    
    return loss_mu + loss_var + (0.01 * kl_loss)

# --- 4. DATASET & LOADING ---
class UnsupervisedGraphDataset(Dataset):
    def __init__(self, file_list, gvae_model, device):
        super().__init__()
        self.file_list = file_list
        self.gvae_model = gvae_model
        self.device = device
        
        # Freeze GVAE
        self.gvae_model.eval()
        for param in self.gvae_model.parameters():
            param.requires_grad = False

    def len(self):
        return len(self.file_list)

    def get(self, idx):
        path = self.file_list[idx]
        data = torch.load(path, weights_only=False).to(self.device)
        
        # Transform Raw Graph -> Spatial Embeddings
        with torch.no_grad():
            z_nodes = self.gvae_model.encode(data.x, data.edge_index)
        
        # Return Spatial Z + Original One-Hot (data.x)
        return Data(
            z_spatial=z_nodes.cpu(), 
            x_onehot=data.x.cpu(),
            num_nodes=data.num_nodes
        )

# --- 5. EXECUTION PIPELINE ---

# A. Load Frozen GVAE
print("❄️ Loading Frozen GVAE...")
gvae_encoder = VariationalGATEncoder(159, 64, 16) 
gvae_model = VGAE(gvae_encoder).to(DEVICE)
gvae_model.load_state_dict(torch.load(GVAE_PATH, map_location=DEVICE, weights_only=True))
gvae_model.eval()

# B. Prepare Data
print("📦 Preparing Data...")
files = glob.glob(f"{OUTPUT_DIR}/*.pt")
valid_files = [f for f in files if os.path.basename(f).replace('.pt', '') in adata.obs['condition'].to_dict()]

train_files, test_files = train_test_split(valid_files, test_size=0.1, random_state=42)
train_ds = UnsupervisedGraphDataset(train_files, gvae_model, DEVICE)
test_ds = UnsupervisedGraphDataset(test_files, gvae_model, DEVICE)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

# C. Train Moment Matching VAE
model = MomentMatchingVAE().to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LR)

print(f"\n🔄 Training Moment-Matching VAE for {EPOCHS} Epochs...")
best_val_loss = float('inf')
patience_counter = 0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=True)
    
    for batch in loop:
        batch = batch.to(DEVICE)
        optimizer.zero_grad()
        
        # Forward
        z_cell, p_means, p_vars, mu, logstd = model(batch.z_spatial, batch.x_onehot, batch.batch)
        
        # Loss
        loss = moment_loss(p_means, p_vars, batch.z_spatial, batch.x_onehot, batch.batch, mu, logstd)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())
    
    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(DEVICE)
            z_cell, p_means, p_vars, mu, logstd = model(batch.z_spatial, batch.x_onehot, batch.batch)
            v_loss = moment_loss(p_means, p_vars, batch.z_spatial, batch.x_onehot, batch.batch, mu, logstd)
            val_loss += v_loss.item()
    
    avg_train_loss = total_loss / len(train_loader)
    avg_val_loss = val_loss / len(test_loader)
    
    print(f"   Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    
    # --- EARLY STOPPING CHECK ---
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), os.path.join(CACHE_DIR, "best_moment_vae.pth"))
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"🛑 Early stopping triggered after {epoch+1} epochs.")
            break

print("✅ Training Complete.")

## LOAD THE MODEL

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GlobalAttention, GATv2Conv, VGAE
from torch_geometric.utils import scatter
import os
import glob
import numpy as np

# --- CONFIGURATION ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_PROTEINS = 159      # Total protein types (Check if 158 or 159 in your data!)
INPUT_DIM = 16          # Latent dim from Frozen GVAE
HIDDEN_DIM = 128
LATENT_DIM = 32         # Cell State Embedding
HEADS = 4

# Paths
BASE_DIR = "/home/projects/nyosef/zvise/PixelGen/PixelGen/cache/GVAE"
GVAE_PATH = os.path.join(BASE_DIR, "models/gvae_gat_model.pth")
MOMENT_VAE_PATH = os.path.join(BASE_DIR, "moment_matching_vae/best_moment_vae.pth")
GRAPHS_DIR = "/home/projects/nyosef/zvise/PixelGen/PixelGen/cache/GVAE/graphs" # Raw graphs

# --- ARCHITECTURES ---
class VariationalGATEncoder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GATv2Conv(in_channels, hidden_channels, heads=HEADS, concat=True)
        hidden_out = hidden_channels * HEADS
        self.conv_mu = GATv2Conv(hidden_out, out_channels, heads=1, concat=False)
        self.conv_logstd = GATv2Conv(hidden_out, out_channels, heads=1, concat=False)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        return self.conv_mu(x, edge_index), self.conv_logstd(x, edge_index)

class MomentMatchingVAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.prot_encoder = nn.Linear(NUM_PROTEINS, 16)
        fusion_dim = INPUT_DIM + 16
        
        self.att_gate = nn.Sequential(nn.Linear(fusion_dim, 64), nn.Tanh(), nn.Linear(64, 1))
        self.pool = GlobalAttention(gate_nn=self.att_gate)
        
        self.enc_fc = nn.Sequential(nn.Linear(fusion_dim, HIDDEN_DIM), nn.ReLU(), nn.BatchNorm1d(HIDDEN_DIM))
        self.fc_mu = nn.Linear(HIDDEN_DIM, LATENT_DIM)
        self.fc_var = nn.Linear(HIDDEN_DIM, LATENT_DIM)
        
        self.decoder_mu = nn.Sequential(nn.Linear(LATENT_DIM, HIDDEN_DIM), nn.ReLU(), nn.Linear(HIDDEN_DIM, NUM_PROTEINS * INPUT_DIM))
        self.decoder_var = nn.Sequential(nn.Linear(LATENT_DIM, HIDDEN_DIM), nn.ReLU(), nn.Linear(HIDDEN_DIM, NUM_PROTEINS * INPUT_DIM), nn.Softplus())

    def encode(self, z_spatial, x_onehot, batch):
        h_prot = F.relu(self.prot_encoder(x_onehot))
        h_fused = torch.cat([z_spatial, h_prot], dim=1)
        graph_h = self.pool(h_fused, batch)
        h = self.enc_fc(graph_h)
        return self.fc_mu(h), self.fc_var(h)

    def reparameterize(self, mu, logstd):
        return mu # Deterministic for evaluation

    def forward(self, z_spatial, x_onehot, batch):
        mu, logstd = self.encode(z_spatial, x_onehot, batch)
        z_cell = self.reparameterize(mu, logstd)
        pred_means = self.decoder_mu(z_cell).view(-1, NUM_PROTEINS, INPUT_DIM)
        pred_vars = self.decoder_var(z_cell).view(-1, NUM_PROTEINS, INPUT_DIM)
        return z_cell, pred_means, pred_vars

# --- LOADING ---
print("🔄 Loading Models...")
# 1. GVAE
gvae_encoder = VariationalGATEncoder(NUM_PROTEINS, 64, INPUT_DIM)
gvae_model = VGAE(gvae_encoder).to(DEVICE)
gvae_model.load_state_dict(torch.load(GVAE_PATH, map_location=DEVICE, weights_only=True))
gvae_model.eval()

# 2. Moment VAE
model = MomentMatchingVAE().to(DEVICE)
model.load_state_dict(torch.load(MOMENT_VAE_PATH, map_location=DEVICE, weights_only=True))
model.eval()

print("✅ Models Loaded.")

In [ ]:
model

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import r2_score
from tqdm import tqdm

# --- 1. COLLECT METRICS ---
true_means_list = []
pred_means_list = []
true_vars_list = []
pred_vars_list = []

files = glob.glob(f"{GRAPHS_DIR}/*.pt")
# Grab a random subset for speed (e.g., 200 cells)
test_files = np.random.choice(files, size=min(200, len(files)), replace=False)

print(f"📊 Evaluating on {len(test_files)} cells...")

with torch.no_grad():
    for f in tqdm(test_files):
        try:
            # Load & Prep
            data = torch.load(f, weights_only=False).to(DEVICE)
            batch = torch.zeros(data.num_nodes, dtype=torch.long, device=DEVICE) # Single graph batch
            
            # Run GVAE (Get Ground Truth Latents)
            z_spatial = gvae_model.encode(data.x, data.edge_index)
            
            # Run Moment VAE (Get Predictions)
            _, pred_mu, pred_sigma = model(z_spatial, data.x, batch)
            
            # --- CALCULATE GROUND TRUTH MOMENTS (Same logic as Loss) ---
            prot_ids = data.x.argmax(dim=1)
            # Filter: Only look at proteins present in this cell
            present_prots = torch.unique(prot_ids)
            
            for pid in present_prots:
                # Mask for this protein
                mask = (prot_ids == pid)
                if mask.sum() < 2: continue # Need >1 node for variance
                
                # Ground Truth for this protein
                z_prot = z_spatial[mask]
                true_mu = z_prot.mean(dim=0)
                true_var = z_prot.var(dim=0, unbiased=False) # Biased to match scatter reduce logic
                
                # Prediction for this protein
                # pred_mu is shape [1, 159, 16] -> [159, 16]
                p_mu = pred_mu[0, pid]
                p_var = pred_sigma[0, pid]
                
                true_means_list.append(true_mu.cpu().numpy())
                pred_means_list.append(p_mu.cpu().numpy())
                true_vars_list.append(true_var.cpu().numpy())
                pred_vars_list.append(p_var.cpu().numpy())

        except Exception as e:
            continue

# Flatten lists
Y_true_mean = np.concatenate(true_means_list).flatten()
Y_pred_mean = np.concatenate(pred_means_list).flatten()
Y_true_var = np.concatenate(true_vars_list).flatten()
Y_pred_var = np.concatenate(pred_vars_list).flatten()

# --- 2. PLOTTING ---
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Plot A: Mean Reconstruction (Location)
r2_mean = r2_score(Y_true_mean, Y_pred_mean)
sns.scatterplot(x=Y_true_mean, y=Y_pred_mean, ax=axes[0], s=5, alpha=0.3, color='#1f77b4')
# Perfect line
min_val, max_val = Y_true_mean.min(), Y_true_mean.max()
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2)
axes[0].set_title(f"Reconstruction of Protein Locations (Means)\n$R^2 = {r2_mean:.3f}$", fontsize=14)
axes[0].set_xlabel("True Mean (Latent Space)", fontsize=12)
axes[0].set_ylabel("Predicted Mean (Latent Space)", fontsize=12)

# Plot B: Variance Reconstruction (Shape/Spread)
# Log scale often helps visual variance better
r2_var = r2_score(Y_true_var, Y_pred_var)
sns.scatterplot(x=Y_true_var, y=Y_pred_var, ax=axes[1], s=5, alpha=0.3, color='#ff7f0e')
min_val, max_val = Y_true_var.min(), Y_true_var.max()
axes[1].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2)
axes[1].set_title(f"Reconstruction of Protein Distribution (Variances)\n$R^2 = {r2_var:.3f}$", fontsize=14)
axes[1].set_xlabel("True Variance", fontsize=12)
axes[1].set_ylabel("Predicted Variance", fontsize=12)
axes[1].set_xscale('log')
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import umap
import matplotlib.pyplot as plt
import seaborn as sns

def extract_moment_embeddings(model, files, gvae_model, device):
    """
    Runs inference on a list of files to get the VAE Latent Code (z_cell).
    """
    model.eval()
    embeddings = []
    cell_ids = []
    
    # Create a loader for ALL files (batch size can be higher for inference)
    # We reuse the dataset class you defined in the training block
    ds = UnsupervisedGraphDataset(files, gvae_model, device)
    loader = DataLoader(ds, batch_size=16, shuffle=False)
    
    print(f"Extracting embeddings from {len(files)} cells...")
    
    with torch.no_grad():
        for i, batch in enumerate(tqdm(loader)):
            batch = batch.to(device)
            
            # Forward pass (Unsupervised)
            # We only care about the first output: z_cell
            z_cell, _, _, _, _ = model(batch.z_spatial, batch.x_onehot, batch.batch)
            
            embeddings.append(z_cell.cpu().numpy())
            
            # Track Cell IDs to map back to labels later
            # Indices [i*batch_size : (i+1)*batch_size]
            start = i * loader.batch_size
            end = start + batch.num_graphs
            batch_files = files[start:end]
            batch_ids = [os.path.basename(f).replace('.pt', '') for f in batch_files]
            cell_ids.extend(batch_ids)

    return np.vstack(embeddings), cell_ids

# --- 1. RUN EXTRACTION ---
# Reuse 'valid_files' from your training script so we have labels
# valid_files was defined in your script as:
# valid_files = [f for f in files if os.path.basename(f)... in adata.obs...]

latent_vectors, ids = extract_moment_embeddings(model, valid_files, gvae_model, DEVICE)

print(f"✅ Extracted shape: {latent_vectors.shape}") 
# Should be (Num_Cells, 32)

# --- 2. MAP LABELS & VISUALIZE ---
# We need to look up the condition (Control/4h/24h) for each Cell ID
id_to_cond = adata.obs['condition'].to_dict()
labels = [id_to_cond.get(cid, "Unknown") for cid in ids]

# Create DataFrame
df_embed = pd.DataFrame(latent_vectors, columns=[f"Z_{i}" for i in range(latent_vectors.shape[1])])
df_embed['Condition'] = labels



In [ ]:
df_latent = pd.DataFrame(
    latent_vectors, 
    index=ids, 
    columns=[f"z_moment_{i}" for i in range(latent_vectors.shape[1])]
)
df_latent

In [ ]:

# 2. Reorder df_latent to match adata_vae exactly
# (This is critical: adata.obsm assumes the rows are in the exact same order)
df_latent_ordered = df_latent.loc[adata.obs_names]

# 3. Assign to obsm
adata.obsm['gvae_attention_embedding'] = df_latent_ordered.values



In [ ]:
sc.pp.neighbors(
    adata,
    use_rep="gvae_attention_embedding",
    n_neighbors=15
)

sc.tl.umap(adata)

In [ ]:
sc.pl.umap(
    adata,
    color=['condition','cell_type','cell_group'],
    frameon=False,
    size=40
)